# Quickstart: Three-axis pitch analysis of *Pierrot lunaire* No. 7

This notebook demonstrates the complete pipeline for analyzing Sprechstimme
pitch in Schoenberg's *Pierrot lunaire* op.21, No. 7 ("Der kranke Mond").

## Overview

The analysis decomposes each performer's pitch relative to the score into
three independent axes:

- **register** (offset): overall pitch shift (in cents)
- **range** (compression): pitch span expansion / compression ratio
- **contour** (direction): adherence to score pitch shape (Spearman r)

We classify performances along the *score-faithful ↔ directed-recitation ↔ dynamic* spectrum.

## Data

- **Audio**: Stiedry-Wagner 1940 recording (fetched from archive.org)
- **Metadata**: segments.csv, score_events.csv, segment_score_map.csv (bundled)

⚠️ **Before running**: Please read [`LEGAL_NOTICE.md`](../LEGAL_NOTICE.md).

## 1. Setup

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add package to path (in case not installed)
pkg_path = Path('..').resolve() / 'src'
if pkg_path not in sys.path:
    sys.path.insert(0, str(pkg_path))

import sprechstimme_pitch
from sprechstimme_pitch import metrics, pitch, plotting

print(f"Package version: {sprechstimme_pitch.__version__}")

# Set paths
REPO_ROOT = Path('..').resolve()
AUDIO_DIR = REPO_ROOT / 'data' / 'audio'
METADATA_DIR = REPO_ROOT / 'data' / 'metadata'
EXPORT_DIR = REPO_ROOT / 'outputs'
EXPORT_DIR.mkdir(exist_ok=True)

print(f"\nPaths:")
print(f"  Audio: {AUDIO_DIR}")
print(f"  Metadata: {METADATA_DIR}")
print(f"  Outputs: {EXPORT_DIR}")

## 2. Fetch audio (if needed)

In [ ]:
import subprocess

audio_file = list(AUDIO_DIR.glob('*.mp3')) + list(AUDIO_DIR.glob('*.flac'))

if not audio_file:
    print("No audio found. Fetching from archive.org...")
    print("\n⚠️  READ LEGAL_NOTICE.md FIRST!\n")
    
    result = subprocess.run(
        ['python', str(REPO_ROOT / 'scripts' / 'fetch_audio.py'), '--yes'],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
    audio_file = list(AUDIO_DIR.glob('*.mp3')) + list(AUDIO_DIR.glob('*.flac'))

if audio_file:
    print(f"✓ Audio file: {audio_file[0].name}")
else:
    raise FileNotFoundError(f"No audio in {AUDIO_DIR}")

## 3. Load metadata

In [ ]:
# Load metadata CSVs
segments_df = pd.read_csv(METADATA_DIR / 'segments.csv')
score_events_df = pd.read_csv(METADATA_DIR / 'score_events.csv')
segment_score_map_df = pd.read_csv(METADATA_DIR / 'segment_score_map.csv')

print("Segments (sample):")
print(segments_df.head(2))
n_recs = segments_df['recording_id'].nunique()
print(f"\nTotal: {len(segments_df)} segments across {n_recs} recordings")

print("\nScore events (sample):")
print(score_events_df.head(2))

print("\nSegment-score map (sample):")
print(segment_score_map_df.head(3))


## 4. Pitch tracking (pYIN)

In [ ]:
# Load audio — resampled to 22.05 kHz mono to match the paper-1 pYIN config.
import librosa

audio_file = (list(AUDIO_DIR.glob('*.mp3')) + list(AUDIO_DIR.glob('*.flac')))[0]
sr_pyin = 22050
y, sr = librosa.load(audio_file, sr=sr_pyin, mono=True)

print(f"Audio loaded: {audio_file.name}")
print(f"  Sample rate: {sr} Hz")
print(f"  Duration: {len(y) / sr:.1f} s")

# Pitch tracking
print("\nRunning pYIN pitch tracking...")
pitch_track = pitch.track_pitch(y, sr, fmin=130.8, fmax=523.3)

print(f"✓ Tracked {len(pitch_track.f0_hz)} frames")
voiced_ratio = float(np.nanmean(pitch_track.voiced_flag))
print(f"  Voicing ratio: {voiced_ratio:.1%}")
print(f"  F0 range: {np.nanmin(pitch_track.f0_hz):.1f} — {np.nanmax(pitch_track.f0_hz):.1f} Hz")

## 5. Segment extraction and alignment

In [ ]:
# Demo: analyze one voice segment of the recording whose audio we have.
# fetch_audio.py provides the Stiedry-Wagner 1940 recording (sch-1940-sti);
# this demo uses its m5 segment. A full analysis iterates over all segments.
DEMO_RECORDING = 'sch-1940-sti'
DEMO_SEGMENT = 'seg_p07_m5'

seg = segments_df[
    (segments_df['recording_id'] == DEMO_RECORDING)
    & (segments_df['segment_id'] == DEMO_SEGMENT)
].iloc[0]
recording_id = seg['recording_id']
segment_id = seg['segment_id']
piece_id = seg['piece_id']

start_s, end_s = float(seg['start_s']), float(seg['end_s'])
hop_length = 256
sr_pyin = 22050

# Extract segment from full pitch track
start_frame = int(start_s * sr_pyin / hop_length)
end_frame = int(end_s * sr_pyin / hop_length)
f0_segment = pitch_track.f0_hz[start_frame:end_frame]
voiced_segment = pitch_track.voiced_flag[start_frame:end_frame]

# Convert to cents (A4 = 440 Hz = 6900 cents MIDI)
f0_cent = 1200 * np.log2(f0_segment / 440.0) + 6900

# Get mapped score events for this segment
segment_map = segment_score_map_df[
    (segment_score_map_df['recording_id'] == recording_id)
    & (segment_score_map_df['segment_id'] == segment_id)
].copy()

print(f"Analyzing: {recording_id} / {segment_id}")
print(f"  Segment: {start_s:.1f} — {end_s:.1f} s")
print(f"  Frames: {len(f0_segment)}")
print(f"  Mapped notes: {len(segment_map)}")
print("\nSegment-score mapping (first 3 notes):")
print(segment_map[['bar_number', 'note_index', 'start_s', 'end_s']].head(3))

## 6. Extract per-note pitch and score reference

In [ ]:
# For each note in the mapping, extract pitch + reliability diagnostics.
hop_s = hop_length / sr_pyin  # seconds per frame

note_data = []
for _, note_row in segment_map.iterrows():
    note_start_s = float(note_row['start_s'])
    note_end_s = float(note_row['end_s'])

    # Frame indices relative to the full track
    note_start_frame = int(note_start_s / hop_s)
    note_end_frame = int(note_end_s / hop_s)
    note_start_frame = max(0, min(note_start_frame, len(pitch_track.f0_hz) - 1))
    note_end_frame = max(note_start_frame + 1, min(note_end_frame, len(pitch_track.f0_hz)))

    # Per-note pYIN slice
    note_f0 = pitch_track.f0_hz[note_start_frame:note_end_frame]
    note_voiced = pitch_track.voiced_flag[note_start_frame:note_end_frame]

    # Convert to cents (relative to MIDI 69 = A4 = 6900 cents)
    note_f0_cent = 1200 * np.log2(note_f0 / 440.0) + 6900
    note_f0_median = (
        float(np.nanmedian(note_f0_cent)) if np.any(~np.isnan(note_f0_cent)) else np.nan
    )

    # Reliability diagnostics
    voiced_ratio = pitch.note_voiced_ratio(note_voiced)
    f0_iqr_cent = pitch.note_f0_iqr_cent(note_f0, note_voiced)

    # Score reference
    score_match = score_events_df[
        (score_events_df['piece_id'] == piece_id)
        & (score_events_df['bar_number'] == int(note_row['bar_number']))
        & (score_events_df['note_index'] == int(note_row['note_index']))
    ]
    ref_cent = (
        float(score_match.iloc[0]['ref_pitch_cent']) if len(score_match) > 0 else np.nan
    )

    err_cent = (
        note_f0_median - ref_cent
        if not (np.isnan(note_f0_median) or np.isnan(ref_cent))
        else np.nan
    )

    note_data.append({
        'bar': int(note_row['bar_number']),
        'note': int(note_row['note_index']),
        'est_cent': note_f0_median,
        'ref_cent': ref_cent,
        'err_cent': err_cent,
        'voiced_ratio': voiced_ratio,
        'f0_iqr_cent': f0_iqr_cent,
    })

notes_df = pd.DataFrame(note_data)
print(f"Extracted {len(notes_df)} notes with diagnostics:")
print(notes_df.head(10))


## 7. Compute three-axis metrics

In [ ]:
# Apply the issue-#12 reliability spec note-by-note:
#   voiced_low OR iqr_high OR pitch-class subharmonic error.
unreliable, reasons = [], []
for _, r in notes_df.iterrows():
    pc_err = pitch.classify_pitch_class_error(r['err_cent'])
    flag, reason_str = pitch.is_pyin_unreliable(
        voiced_ratio=r['voiced_ratio'],
        f0_iqr_cent=r['f0_iqr_cent'],
        pitch_class_error=pc_err,
    )
    unreliable.append(flag)
    reasons.append(reason_str)

notes_df['is_pyin_unreliable'] = unreliable
notes_df['unreliable_reasons'] = reasons

n_unreliable = int(sum(unreliable))
print(f"Unreliable notes: {n_unreliable} / {len(notes_df)}")
if n_unreliable:
    print(notes_df.loc[notes_df['is_pyin_unreliable'], ['bar', 'note', 'unreliable_reasons']])

# Compute the three-axis metrics on the reliable subset
m = metrics.compute_three_axis_metrics(
    est_cent=notes_df['est_cent'].to_numpy(),
    score_cent=notes_df['ref_cent'].to_numpy(),
    unreliable_flags=notes_df['is_pyin_unreliable'].to_numpy(),
    min_notes=3,
)

print("\nThree-axis metrics for this segment:")
print(f"  Register offset:     {m.register_offset_cent:+7.1f} cents")
print(f"  Range compression:   {m.range_compression:7.2f} (std obs / std score)")
print(f"  Contour correlation: {m.contour_correlation:7.2f} (Spearman r)")
print(f"  Notes used:          {m.n_notes_used} / {len(notes_df)}")

# Type classification needs per-recording aggregates (multi-segment).
# A single segment cannot produce contour_std, so the result below is
# illustrative only; in the paper-1 pipeline, classify_performance is
# called after aggregate_metrics() across all four voice segments.
perf_type = metrics.classify_performance(
    register_offset_cent=m.register_offset_cent,
    contour_std=0.0,
)
print(f"\n  -> Type (illustrative; needs multi-segment input): {perf_type}")


## 8. Visualization (example: radar chart)

In [ ]:
# For a complete demo, we would compute metrics for all recordings.
# Here we show how to visualize a single recording.

# Normalize axes for radar
axes_norm = plotting.four_axes_normalize(
    register_offset_cent=m.register_offset_cent,
    range_compression=m.range_compression,
    contour_correlation=m.contour_correlation,
    contour_std=0.1,  # Example value
)

print("Normalized axes for radar chart:")
for key, val in axes_norm.items():
    print(f"  {key:20s}: {val:.3f}")

# Create a minimal example visualization
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))

axis_keys = ['abs_offset', 'range_dev', 'contour_dev', 'contour_std']
axis_labels = ['|offset|/1000', '|1-range|', '1-contour', 'contour_std']
values = [axes_norm.get(k, 0) for k in axis_keys]
values += values[:1]  # Close polygon

angles = np.linspace(0, 2 * np.pi, len(axis_keys), endpoint=False).tolist()
angles += angles[:1]

ax.plot(angles, values, 'o-', linewidth=2, label='Stiedry-Wagner 1940')
ax.fill(angles, values, alpha=0.25)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(axis_labels)
ax.set_title('Performance Profile (Demo)', fontsize=12, pad=20)
ax.grid(True)

plt.tight_layout()
plt.savefig(EXPORT_DIR / 'radar_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {EXPORT_DIR / 'radar_demo.png'}")

## Summary

This notebook demonstrated the core pipeline:

1. **Load audio** -> pYIN pitch tracking
2. **Align segments** -> map to score events by duration weighting
3. **Extract per-note pitch** -> compute frame statistics + per-note voiced ratio and IQR
4. **Flag unreliable estimates** -> `is_pyin_unreliable` (voiced_low OR iqr_high OR pitch-class subharmonic)
5. **Compute 3-axis metrics** -> register / range (std obs / std score) / contour
6. **Aggregate across segments** then **classify_performance** -> score-faithful / directed-recitation / dynamic
7. **Visualize** -> radar chart, PCA biplot, decision flow

For the full analysis (5 recordings x 4 voice segments), iterate this pipeline
over every (recording, segment) pair and pass the per-segment metrics through
`metrics.aggregate_metrics` before calling `metrics.classify_performance`.
